In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

from typing import Union, Tuple, Optional, List

def download_dataset(
    name: str = "flwrlabs/celeba",
    cache_dir: Optional[str] = "./dataset"
    ):

    from datasets import load_dataset
    dataset = load_dataset(name,cache_dir=cache_dir)

    print(dataset)

    print("Train:", len(dataset["train"]))
    print("Valid:", len(dataset["valid"]))
    print("Test :", len(dataset["test"]))

    total = sum(len(dataset[x]) for x in dataset)
    print("Total:", total)

    return dataset


class CelebADataset(Dataset):
    def __init__(
        self,
        hf_dataset,
        img_size: Union[int, Tuple[int, int]],
        attributes: Optional[List[str]] = None
    ):
        self.dataset = hf_dataset

        if isinstance(img_size, tuple):
            self.img_size = img_size
        else:
            self.img_size = (img_size, img_size)

        self.transform = transforms.Compose([
            transforms.Resize(self.img_size),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

        if attributes is not None:
            self.attributes = attributes
        else:
            all_features = list(self.dataset.features.keys())
            not_feature = ["image", "celeb_id"]
            self.attributes = [f for f in all_features if f not in not_feature]

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img = self.dataset[idx]["image"]
        if not isinstance(img, Image.Image):
            img = Image.fromarray(img.numpy(), mode="RGB")
        img = self.transform(img)

        attr_tensor = torch.tensor(
            [self.dataset[idx][attr] for attr in self.attributes],
            dtype=torch.float32
        )
        return img, attr_tensor
import torch
from torch.utils.data import Dataset, DataLoader, Subset
import random

def dataloaders(
    dataset,
    img_size: int = 128,
    batch_size: int = 32,
    num_workers: int = 4,
    pin_memory: bool = True,
    shuffle: bool = True,
    test_set: bool = True,
    cache_dir: Optional[str] = "./dataset",
    subset_ratio: float = 1.0,
    seed: int = 42
) -> tuple[DataLoader, DataLoader, DataLoader | None]:

    dataset = download_dataset(dataset, cache_dir)

    train_dataset = CelebADataset(dataset["train"], img_size)
    val_dataset = CelebADataset(dataset["valid"], img_size)
    test_dataset = CelebADataset(dataset["test"], img_size) if test_set else None

    def subsample(ds, ratio):
        if ratio >= 1.0:
            return ds
        n = len(ds)
        k = int(n * ratio)
        g = torch.Generator()
        g.manual_seed(seed)
        indices = torch.randperm(n, generator=g)[:k].tolist()
        return Subset(ds, indices)

    train_dataset = subsample(train_dataset, subset_ratio)
    val_dataset = subsample(val_dataset, subset_ratio)
    if test_dataset is not None:
        test_dataset = subsample(test_dataset, subset_ratio)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=pin_memory,
        shuffle=shuffle,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=pin_memory,
        shuffle=False,
    )

    test_loader = None
    if test_dataset is not None:
        test_loader = DataLoader(
            test_dataset,
            batch_size=batch_size,
            num_workers=num_workers,
            pin_memory=pin_memory,
            shuffle=False,
        )

    return train_loader, val_loader, test_loader

def show_images(
    images,
    attributes=None,
    num_cols=4,
    figsize=(12, 12)
    ):
    """
    images: torch.Tensor of shape (B, C, H, W), values normalized [-1, 1]
    attributes: optional torch.Tensor of shape (B, num_attrs) – for titles
    """
    images = images * 0.5 + 0.5
    images = torch.clamp(images, 0, 1)

    batch_size = images.size(0)
    num_rows = (batch_size + num_cols - 1) // num_cols

    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(num_rows, num_cols, figsize=figsize)
    axes = axes.flatten()

    for i in range(batch_size):
        img = images[i].permute(1, 2, 0).cpu().numpy()
        axes[i].imshow(img)
        axes[i].axis('off')
        if attributes is not None:
            attrs = attributes[i]
            # For brevity, just show count of positive attributes
            pos_count = (attrs > 0.5).sum().item()
            axes[i].set_title(f"Pos: {pos_count}", fontsize=8)

    for j in range(batch_size, len(axes)):
        axes[j].axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(
        self,
        input_channels: int = 3,
        latent_dim: int = 128,
        hidden_dims: list = [32, 64, 128, 256, 512],
        img_size: int = 128,
        cond_dim: int = 0
    ):
        super().__init__()
        self.latent_dim = latent_dim
        self.cond_dim = cond_dim

        modules = []
        in_channels = input_channels
        for h_dim in hidden_dims:
            modules.append(
                nn.Sequential(
                    nn.Conv2d(in_channels, h_dim, kernel_size=3, stride=2, padding=1),
                    nn.BatchNorm2d(h_dim),
                    nn.LeakyReLU(negative_slope=0.2, inplace=True)
                )
            )
            in_channels = h_dim
        self.encoder = nn.Sequential(*modules)

        dummy_input = torch.zeros(1, input_channels, img_size, img_size)
        with torch.no_grad():
            dummy_output = self.encoder(dummy_input)
            self.flatten_dim = dummy_output.view(1, -1).size(1)

        self.fc_mu = nn.Linear(self.flatten_dim + cond_dim, latent_dim)
        self.fc_log_var = nn.Linear(self.flatten_dim + cond_dim, latent_dim)

        nn.init.constant_(self.fc_log_var.bias, 0.0)
        nn.init.normal_(self.fc_log_var.weight, mean=0.0, std=0.01)

        nn.init.xavier_normal_(self.fc_mu.weight)
        nn.init.constant_(self.fc_mu.bias, 0.0)

    @staticmethod
    def reparameterization_trick(mu, log_var):
        std = torch.exp(log_var * 0.5)
        eps = torch.randn_like(std)
        return mu + (std * eps)

    def encode(self, x, c=None):
        h = self.encoder(x)
        h = h.view(h.size(0), -1)

        if c is not None:
            h = torch.cat([h, c], dim=1)

        mu = self.fc_mu(h)
        log_var = self.fc_log_var(h)
        return mu, log_var

    def forward(self, x, c=None):
        mu, log_var = self.encode(x, c)
        z = self.reparameterization_trick(mu, log_var)
        return z, mu, log_var

In [ ]:
import torch
import torch.nn as nn

class Decoder(nn.Module):
    def __init__(
        self,
        latent_dim: int,
        hidden_dims: list,
        output_channels: int,
        spatial_shape: tuple
    ):
        super().__init__()

        channels, height, width = spatial_shape
        self.hidden_dim = channels
        self.spatial_height = height
        self.spatial_width = width
        self.flatten_dim = channels * height * width

        self.fc = nn.Linear(latent_dim, self.flatten_dim)

        modules = []
        in_channels = self.hidden_dim
        for h_dim in reversed(hidden_dims):
            modules.append(
                nn.Sequential(
                    nn.ConvTranspose2d(
                        in_channels, h_dim,
                        kernel_size=3, stride=2, padding=1, output_padding=1
                    ),
                    nn.BatchNorm2d(h_dim),
                    nn.LeakyReLU(negative_slope=0.2, inplace=True)
                )
            )
            in_channels = h_dim

        """
        We use Tanh() because images are normalized in range of [-1,+1]
        If we use Sigmoid(), we should normalize in range of [0,+1]
        """
        modules.append(
            nn.Sequential(
                nn.Conv2d(in_channels, output_channels, kernel_size=3, padding=1),
                nn.Tanh()
            )
        )
        self.decoder = nn.Sequential(*modules)

    def forward(self, z):
        h = self.fc(z)
        h = h.view(-1, self.hidden_dim, self.spatial_height, self.spatial_width)
        return self.decoder(h)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class VAE(nn.Module):
    def __init__(
        self,
        input_channels: int = 3,
        latent_dim: int = 128,
        hidden_dims: list = [32, 64, 128, 256, 512],
        img_size: int = 128
    ):
        super().__init__()
        self.latent_dim = latent_dim

        self.encoder = Encoder(input_channels, latent_dim, hidden_dims, img_size)

        dummy_input = torch.zeros(1, input_channels, img_size, img_size)
        with torch.no_grad():

            conv_output = self.encoder.encoder(dummy_input)
            _, channels, height, width = conv_output.shape
            spatial_shape = (channels, height, width)
            print(f"Dynamically detected encoder output shape: {spatial_shape}")

        self.decoder = Decoder(
            latent_dim=latent_dim,
            hidden_dims=hidden_dims,
            output_channels=input_channels,
            spatial_shape=spatial_shape
        )

    @staticmethod
    def reparameterization_trick(
        mu,
        log_var
    ):
        std = torch.exp(log_var * 0.5)
        eps = torch.randn_like(std)
        return mu + (std * eps)

    def forward(
        self,
        x
    ):
        mu, log_var = self.encoder.encode(x)

        if self.training:
            z = self.reparameterization_trick(mu, log_var)
        else:
            z = mu

        x_recon = self.decoder(z)

        return x_recon, mu, log_var

    def loss(self, recon_x, x, mu, log_var, beta=1.0):
        """
        Args:
            recon_x: reconstructed images (output of decoder)
            x: original input images
            mu, log_var: latent statistics from encoder
            beta: weight for KL divergence (β-VAE).
                  For 128x128 images, start with beta=0.0005.

        Returns:
            total_loss: scalar tensor (recon + beta * kl)
            recon_loss: per-pixel MSE (mean over pixels and batch) – scale ~0.0–1.0
            kl_loss: KL divergence (summed over latent dims, averaged over batch) – scale ~0–1000
        """

        # This gives an average error per pixel in the [-1,1] range.
        # Typical values: start ~0.4, end ~0.02–0.05.
        recon_loss = F.mse_loss(recon_x, x, reduction='mean')

        # KL = -0.5 * sum(1 + log_var - mu^2 - exp(log_var))
        # Sum over latent dimensions, then average over batch.
        kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp(), dim=1)
        kl_loss = kl_loss.mean()  # scalar

        # β controls the trade-off. For 128x128 images with per‑pixel MSE,
        # beta=0.0005 is a great starting point.
        total_loss = recon_loss + beta * kl_loss

        return total_loss, recon_loss, kl_loss

    def generate(
        self,
        num_samples,
        device
    ):
        z = torch.randn(num_samples, self.latent_dim).to(device)
        with torch.no_grad():
            return self.decoder(z)

In [ ]:
import os
import yaml
import inspect
from typing import Optional
from collections import deque

import torch
import torch.optim as optim
from torchvision.utils import save_image


def train(
    dataset: str = "flwrlabs/celeba",
    cache_dir: Optional[str] = "./dataset",
    subset_ratio: float = 0.3,
    n_epochs: int = 100,
    img_size: int = 128,
    batch_size: int = 32,
    num_workers: int = 4,
    pin_memory: bool = True,
    shuffle: bool = True,
    test_set: bool = False,
    checkpoint_dir: str = "./checkpoints",
    sample_dir: str = "./outputs/samples",
    input_channels: int = 3,
    latent_dim: int = 128,
    hidden_dims: list = [32, 64, 128, 256, 512],
    learning_rate: float = 2e-4,
    beta: float = 0.0005,
    clip_grad: bool = False,

    kl_plateau_check_start: int = 25,   # start checking a few epochs after annealing ends (epoch 20)
    kl_plateau_window: int = 10,        # compare KL now vs KL this many epochs ago
    kl_plateau_min_drop_frac: float = 0.05,  # require at least 5% relative drop over the window

    use_capacity_annealing: bool = False,
    capacity_target: float = 25.0,      # target nats of KL to allow through, tune per latent_dim
    capacity_anneal_epochs: int = 40,   # ramp epochs, usually longer than beta annealing
    capacity_gamma: float = 100.0,      # weight on |KL - C|, kept large & fixed
):

    device = "cuda" if torch.cuda.is_available() else \
             "mps" if torch.mps.is_available() else \
             "cpu"

    os.makedirs(checkpoint_dir, exist_ok=True)
    os.makedirs(sample_dir, exist_ok=True)

    model = VAE(
        input_channels=input_channels,
        latent_dim=latent_dim,
        hidden_dims=hidden_dims,
        img_size=img_size
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    def get_beta(
        epoch: int,
        max_annealing_epochs: int = 20,
        target_beta: float = 0.0005,
        start_beta_from_zero: bool = False,
    ) -> float:
        if epoch >= max_annealing_epochs:
            return target_beta
        if start_beta_from_zero:
            return target_beta * (epoch / max_annealing_epochs)
        else:
            return target_beta * ((epoch + 1) / max_annealing_epochs)

    def get_capacity(epoch: int) -> float:
        """Linearly ramp allowed KL capacity from 0 to capacity_target."""
        if epoch >= capacity_anneal_epochs:
            return capacity_target
        return capacity_target * (epoch / capacity_anneal_epochs)

    train_loader, val_loader, test_loader = dataloaders(
        dataset=dataset,
        img_size=img_size,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=pin_memory,
        shuffle=shuffle,
        test_set=test_set,
        cache_dir=cache_dir,
        subset_ratio=subset_ratio
    )

    def compute_loss(recon_x, data, mu, log_var, current_beta, current_capacity):
        """Wraps model.loss(); either standard beta-VAE or capacity-annealed KL."""
        _, recon_loss, kl_loss = model.loss(recon_x, data, mu, log_var, beta=0.0)
        # model.loss with beta=0 still gives us recon_loss and kl_loss separately;
        # we build total_loss ourselves depending on the mode.
        if use_capacity_annealing:
            total_loss = recon_loss + capacity_gamma * (kl_loss - current_capacity).abs()
        else:
            total_loss = recon_loss + current_beta * kl_loss
        return total_loss, recon_loss, kl_loss

    def train_epoch(epoch, current_beta, current_capacity):
        model.train()
        total_loss = 0
        total_recon = 0
        total_kl = 0

        for batch_idx, (data, _) in enumerate(train_loader):
            data = data.to(device)
            optimizer.zero_grad()

            recon_x, mu, log_var = model(data)
            loss, recon_loss, kl_loss = compute_loss(recon_x, data, mu, log_var, current_beta, current_capacity)

            loss.backward()

            if clip_grad:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            total_loss += loss.item()
            total_recon += recon_loss.item()
            total_kl += kl_loss.item()

            if batch_idx % 100 == 0:
                print(f"Epoch {epoch:03d} [{batch_idx}/{len(train_loader)}] "
                      f"Loss: {loss.item():.4f}  Recon: {recon_loss.item():.4f}  KL: {kl_loss.item():.4f}")

        num_batches = len(train_loader)
        return total_loss / num_batches, total_recon / num_batches, total_kl / num_batches

    def validate_epoch(target_beta, target_capacity):
        model.eval()
        total_loss = 0
        total_recon = 0
        total_kl = 0

        with torch.no_grad():
            for batch_idx, (data, _) in enumerate(val_loader):
                data = data.to(device)
                recon_x, mu, log_var = model(data)
                loss, recon_loss, kl_loss = compute_loss(recon_x, data, mu, log_var, target_beta, target_capacity)

                total_loss += loss.item()
                total_recon += recon_loss.item()
                total_kl += kl_loss.item()

        num_batches = len(val_loader)
        return total_loss / num_batches, total_recon / num_batches, total_kl / num_batches

    def save_reconstructions(epoch):
        """Save a grid of original vs reconstructed images for visual inspection."""
        model.eval()
        with torch.no_grad():
            data, _ = next(iter(val_loader))
            data = data.to(device)
            recon, _, _ = model(data)

            n = min(8, data.size(0))
            comparison = torch.cat([data[:n], recon[:n]])
            save_image(comparison, f"{sample_dir}/recon_epoch_{epoch:03d}.png",
                       nrow=n, normalize=True)

    best_val_recon = float('inf')
    kl_history = deque(maxlen=kl_plateau_window + 1)
    kl_warning_issued = False

    for epoch in range(1, n_epochs + 1):
        current_beta = get_beta(epoch - 1, target_beta=beta)
        current_capacity = get_capacity(epoch - 1)

        before_lr = optimizer.param_groups[0]['lr']

        mode_str = f"Capacity: {current_capacity:.2f}" if use_capacity_annealing else f"Beta: {current_beta:.6f}"
        print(f"\n===== Epoch {epoch:03d}/{n_epochs} | LR: {before_lr:.2e} | {mode_str} =====")

        train_loss, train_recon, train_kl = train_epoch(epoch, current_beta, current_capacity)
        print(f"Train  -> Loss: {train_loss:.4f}  Recon: {train_recon:.4f}  KL: {train_kl:.4f}")

        val_loss, val_recon, val_kl = validate_epoch(target_beta=beta, target_capacity=capacity_target)
        print(f"Valid  -> Loss: {val_loss:.4f}  Recon: {val_recon:.4f}  KL: {val_kl:.4f}")

        save_reconstructions(epoch)

        scheduler.step(val_recon)

        after_lr = optimizer.param_groups[0]['lr']
        if after_lr < before_lr:
            print(f"LR reduced: {before_lr:.2e} → {after_lr:.2e}")

        kl_history.append(val_kl)
        if (epoch >= kl_plateau_check_start
                and len(kl_history) > kl_plateau_window
                and not kl_warning_issued):
            kl_then = kl_history[0]
            kl_now = kl_history[-1]
            relative_drop = (kl_then - kl_now) / max(kl_then, 1e-8)

            if relative_drop < kl_plateau_min_drop_frac:
                kl_warning_issued = True
                print(
                    f"\n WARNING: val_kl has not meaningfully decreased over the last "
                    f"{kl_plateau_window} epochs ({kl_then:.1f} → {kl_now:.1f}, "
                    f"{relative_drop*100:.1f}% change).\n"
                    f"    The KL term may be too weak to regularize a {latent_dim}-dim latent space "
                    f"at current beta={beta}.\n"
                    f"    Consider: (a) increasing `beta`, (b) enabling `use_capacity_annealing=True`, "
                    f"or (c) reducing `latent_dim`.\n"
                )

        if val_recon < best_val_recon:
            best_val_recon = val_recon
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_recon': best_val_recon,
                'val_kl': val_kl,
                'use_capacity_annealing': use_capacity_annealing,
            }
            torch.save(checkpoint, f"{checkpoint_dir}/best_model.pt")
            print(f"New best model saved with val_recon = {val_recon:.4f}")

    print("\n===== Generating new images from prior =====")
    model.eval()
    with torch.no_grad():
        generated = model.generate(num_samples=16, device=device)
        save_image(generated, f"{sample_dir}/generated_final.png", nrow=4, normalize=True)
        print(f"Generated images saved to {sample_dir}/generated_final.png")


def train_from_yaml(
    yaml_path: str = "./configs/standard_config.yaml"
) -> None:
    with open(yaml_path, 'r') as f:
        config = yaml.safe_load(f)

    sig = inspect.signature(train)
    valid_params = set(sig.parameters.keys())

    filtered_config = {k: v for k, v in config.items() if k in valid_params}

    train(**filtered_config)

In [ ]:
train()

Dynamically detected encoder output shape: (512, 4, 4)


README.md:   0%|          | 0.00/9.28k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

img_align+identity+attr/train-00000-of-0(…): reconstructing file:   0%|          |  0.00B /  500MB            

...

Generating test split:   0%|          | 0/19962 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/19 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'celeb_id', '5_o_Clock_Shadow', 'Arched_Eyebrows', 'Attractive', 'Bags_Under_Eyes', 'Bald', 'Bangs', 'Big_Lips', 'Big_Nose', 'Black_Hair', 'Blond_Hair', 'Blurry', 'Brown_Hair', 'Bushy_Eyebrows', 'Chubby', 'Double_Chin', 'Eyeglasses', 'Goatee', 'Gray_Hair', 'Heavy_Makeup', 'High_Cheekbones', 'Male', 'Mouth_Slightly_Open', 'Mustache', 'Narrow_Eyes', 'No_Beard', 'Oval_Face', 'Pale_Skin', 'Pointy_Nose', 'Receding_Hairline', 'Rosy_Cheeks', 'Sideburns', 'Smiling', 'Straight_Hair', 'Wavy_Hair', 'Wearing_Earrings', 'Wearing_Hat', 'Wearing_Lipstick', 'Wearing_Necklace', 'Wearing_Necktie', 'Young'],
        num_rows: 162770
    })
    valid: Dataset({
        features: ['image', 'celeb_id', '5_o_Clock_Shadow', 'Arched_Eyebrows', 'Attractive', 'Bags_Under_Eyes', 'Bald', 'Bangs', 'Big_Lips', 'Big_Nose', 'Black_Hair', 'Blond_Hair', 'Blurry', 'Brown_Hair', 'Bushy_Eyebrows', 'Chubby', 'Double_Chin', 'Eyeglasses', 'Goatee', 'Gray_Hair', 'H

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch 001 [0/1526] Loss: 0.5349  Recon: 0.5329  KL: 80.5527
Epoch 001 [100/1526] Loss: 0.1096  Recon: 0.1031  KL: 256.6646
Epoch 001 [200/1526] Loss: 0.0833  Recon: 0.0769  KL: 253.2219
Epoch 001 [300/1526] Loss: 0.0718  Recon: 0.0651  KL: 269.5687
Epoch 001 [400/1526] Loss: 0.0668  Recon: 0.0599  KL: 274.1071
Epoch 001 [500/1526] Loss: 0.0645  Recon: 0.0575  KL: 278.4720
Epoch 001 [600/1526] Loss: 0.0599  Recon: 0.0528  KL: 281.7458
Epoch 001 [700/1526] Loss: 0.0546  Recon: 0.0476  KL: 281.7222
Epoch 001 [800/1526] Loss: 0.0577  Recon: 0.0505  KL: 288.2380
Epoch 001 [900/1526] Loss: 0.0496  Recon: 0.0424  KL: 288.8707
Epoch 001 [1000/1526] Loss: 0.0504  Recon: 0.0431  KL: 291.1087
Epoch 001 [1100/1526] Loss: 0.0549  Recon: 0.0478  KL: 282.7122
Epoch 001 [1200/1526] Loss: 0.0608  Recon: 0.0534  KL: 293.0879
Epoch 001 [1300/1526] Loss: 0.0535  Recon: 0.0462  KL: 292.7662
Epoch 001 [1400/1526] Loss: 0.0558  Recon: 0.0483  KL: 300.2470
Epoch 001 [1500/1526] Loss: 0.0451  Recon: 0.0380  KL